In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from catboost import CatBoostRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

In [ ]:
df1 = pd.read_csv("../data/file1.csv")
df1.head()

In [ ]:
df1.tail()

In [ ]:
df1.shape

In [ ]:
df1['filial_created_at'].nunique()


In [ ]:
df1['filial_updated_at'].nunique()

In [ ]:
df1[['created_at', 'filial_created_at', 'filial_updated_at']].head(10)

In [ ]:
df1.isnull().sum()

In [ ]:
df1.describe()

In [ ]:
df1.info()

In [ ]:
df1.nunique()

In [ ]:
new_df1 = df1[["id", "created_at", "filial_created_at", "price", "volume",
               "amount", "dispenser_id", "operator_id",
               "car_id", "temperature", "density", "branch_id",
               "shift_id", "arm_price"]].copy()

In [ ]:
new_df1.head()

In [ ]:
new_df1.shape

In [ ]:
new_df1.isnull().sum()

In [ ]:
new_df1.duplicated().sum()

In [ ]:
new_df1.nunique()

In [ ]:
new_df1["filial_created_at"] = pd.to_datetime(new_df1["filial_created_at"], format='mixed')

In [ ]:
new_df1["created_at"] = pd.to_datetime(new_df1["created_at"], format='mixed')

In [ ]:
new_df1["filial_created_at"].dtype

In [ ]:
new_df1["created_at"].dtype

In [ ]:
farq_soat = (new_df1['created_at'] - new_df1['filial_created_at']).dt.total_seconds() / 3600
print(farq_soat.describe())
print(farq_soat.round().value_counts().head(20))

In [ ]:
new_df1['local_time'] = new_df1['filial_created_at'] + pd.Timedelta(hours=5)

In [ ]:
new_df1['local_time'].dt.floor('s').value_counts().sort_values(ascending=False).head(20)

In [ ]:
new_df1["hour"] = new_df1["local_time"].dt.hour

In [ ]:
new_df1[['local_time', 'hour']].head()

In [ ]:
new_df1["day_of_week"] = new_df1["local_time"].dt.day_name()

In [ ]:
new_df1[['local_time', 'hour', 'day_of_week']].head()

In [ ]:
new_df1['month'] = new_df1['local_time'].dt.month_name()

In [ ]:
new_df1[['local_time', 'hour', 'day_of_week', 'month']].head()

In [ ]:
hourly = new_df1.groupby('hour').agg(
    tranzaksiyalar_soni=('id', 'count'),
    umumiy_hajm=('volume', 'sum')
)

hourly['tranzaksiyalar_soni'].plot( kind='bar', figsize=(12,5), title='Soat bo\'yicha tranzaksiyalar soni')
plt.xlabel('Soat')
plt.ylabel('Tranzaksiyalar soni')
plt.show()

In [ ]:
hourly['umumiy_hajm'].plot( kind='bar', figsize=(12,5), title='Soat bo\'yicha umumiy hajm', color='darkorange')
plt.xlabel('Soat')
plt.ylabel('Umumiy hajm (litr)')
plt.show()

In [ ]:
weekly = new_df1.groupby('day_of_week').agg(
    tranzaksiyalar_soni=('id', 'count'),
    umumiy_hajm=('volume', 'sum')
)

kun_tartibi = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekly = weekly.reindex(kun_tartibi)


weekly['tranzaksiyalar_soni'].plot( kind='bar', figsize=(12,5), title='Hafta kunlari bo\'yicha tranzaksiyalar soni', color='seagreen')
plt.xlabel('Hafta kuni')
plt.ylabel('Tranzaksiyalar soni')
plt.show()

In [ ]:
weekly['umumiy_hajm'].plot(kind='bar', figsize=(12,5), color='pink', title='Hafta kunlari bo\'yicha umumiy hajm')
plt.xlabel('Hafta kuni')
plt.ylabel('Umumiy hajm (litr)')
plt.show()

In [ ]:
monthly = new_df1.groupby('month').agg(
    tranzaksiyalar_soni=('id', 'count'),
    umumiy_hajm=('volume', 'sum')
)
oy_tartibi = ['January', 'February', 'March', 'April', 'May', 'June', 
              'July', 'August', 'September', 'October', 'November', 'December']
monthly = monthly.reindex(oy_tartibi)

monthly['tranzaksiyalar_soni'].plot( kind='bar', figsize=(12,5), title='Oy bo\'yicha tranzaksiyalar soni')
plt.xlabel('Oy')
plt.ylabel('Tranzaksiyalar soni')
plt.show()

In [ ]:
monthly

In [ ]:
new_df1.groupby('month')['local_time'].apply(lambda x: x.dt.date.nunique())

In [ ]:
monthly['umumiy_hajm'].plot(kind='bar', figsize=(12,5), color='pink', title='Oylar bo\'yicha umumiy hajm')
plt.xlabel('Oy nomi')
plt.ylabel('Umumiy hajm (litr)')
plt.show()

## Xulosa: Band/bo'sh vaqtlar tahlili

**Soat bo'yicha:**

Eng band vaqt: ertalab soat 07:00-09:00 oralig'i (peak hour 08:00da, 13547 tranzaksiya)

Eng bo'sh vaqt: kechasi 00:00-04:00 oralig'i (eng kam soat 00:00da, 2895 tranzaksiya)

**Hafta kuni bo'yicha:**

Eng band kun: Seshanba (~49000 tranzaksiya)

Eng bo'sh kun: Yakshanba (~19000 tranzaksiya)

**Oy bo'yicha:**

Eng kop tranzaksiyalar bolgan oy: Iyul ( 41274ta)

Eng kam tranzaksiyalar bolgan oy: May ( 27995 ta, Aprel va Sentyabr oylari toliq berilmagani uchun May oyi tanlandi)


In [ ]:
dispenser_stats = new_df1.groupby('dispenser_id').agg(
    tranzaksiyalar_soni=('id', 'count'),
    umumiy_hajm=('volume', 'sum')
)
dispenser_stats

In [ ]:
dispenser_stats['tranzaksiyalar_soni'].plot( kind='bar', figsize=(12,5), title='Kolonka bo\'yicha tranzaksiyalar soni')
plt.xlabel('Kolonka ID')
plt.ylabel('Tranzaksiyalar soni')
plt.show()

In [ ]:
dispenser_stats['umumiy_hajm'].plot(kind='bar', figsize=(12,5), color='pink', title='Kolonka bo\'yicha umumiy hajm')
plt.xlabel('Kolonka ID')
plt.ylabel('Umumiy hajm')
plt.show()

In [ ]:
new_df1[new_df1["dispenser_id"] == 13]["local_time"].agg(["min", "max"])

In [ ]:
new_df1[new_df1["dispenser_id"] == 14]["local_time"].agg(["min", "max"])

In [ ]:
new_df1[new_df1["dispenser_id"] == 7]["local_time"].agg(["min", "max"])

## Note: **dispenser_id**
Barcha 14 ta kolonka taxminan bir xil davr (5+ oy) davomida ishlagan bo'lsada, kolonka 13 va 14 boshqalariga qaraganda 100+ marta kamroq tranzaksiyaga ega (140 va 1929 ta, boshqalarida 6000-38000 ta). 
Bu farq ishlash muddati bilan bog'liq emas, shuning uchun sababi texnik nosozlik, joylashuv yoki maxsus foydalanish bo'lishi mumkin.

In [ ]:
new_df1['hisoblangan_amount'] = new_df1['price'] * new_df1['volume']
farq = new_df1['amount'] - new_df1['hisoblangan_amount']
farq.describe()

In [ ]:
new_df1.loc[farq.idxmin()]

In [ ]:
new_df1[farq.abs() > 1000].shape

In [ ]:
print(7 / len(new_df1) * 100, "%")

In [ ]:
new_df1[farq.abs() > 1000][['id', 'filial_created_at', 'price', 'volume', 'amount', 'hisoblangan_amount']]

In [ ]:
new_df1[new_df1["volume"] <= 0].shape

In [ ]:
new_df1["volume"].shape

In [ ]:
new_df1[new_df1["amount"] <= 0].shape

In [ ]:
new_df1[new_df1['volume'] <= 0]['volume'].value_counts().head(10)

In [ ]:
new_df1[new_df1['volume'] == 0]['dispenser_id'].value_counts()

In [ ]:
new_df1[new_df1['volume'] == 0]['dispenser_id'].value_counts()

In [ ]:
new_df1_clean = new_df1[(new_df1['volume'] > 0) & (farq.abs() <= 1000)].copy()

In [ ]:
print("Asl qatorlar soni:", len(new_df1))
print("Tozalangandan keyin:", len(new_df1_clean))
print("Chiqarib tashlangan:", len(new_df1) - len(new_df1_clean))

In [ ]:
new_df1_clean.nlargest(10, 'volume')[['id', 'dispenser_id', 'volume', 'amount']]

In [ ]:
new_df1_clean[new_df1_clean['volume'] <= 0].shape
new_df1_clean[new_df1_clean['amount'] <= 0].shape

In [ ]:
new_df1_clean['hisoblangan_amount_v2'] = new_df1_clean['price'] * new_df1_clean['volume']
farq_v2 = new_df1_clean['amount'] - new_df1_clean['hisoblangan_amount_v2']
farq_v2.describe()

In [ ]:
new_df1_clean.columns.to_list()

In [ ]:
kerakli_ustunlar = ['local_time','hour', 'day_of_week', 'month', 'dispenser_id', 'price', 'volume', 'amount', 'branch_id']
df = new_df1_clean[kerakli_ustunlar].copy()

In [ ]:
df = df.sort_values(['dispenser_id', 'local_time'])
df['xizmat_vaqti'] = df.groupby('dispenser_id')['local_time'].diff().dt.total_seconds() / 60

In [ ]:
plt.scatter(df['volume'], df['xizmat_vaqti'], alpha=0.3)
plt.xlabel('Hajm (litr)')
plt.ylabel('Xizmat vaqti (daqiqa)')
plt.show()

In [ ]:
df = df[df['xizmat_vaqti'] < 30]

In [ ]:
plt.scatter(df['volume'], df['xizmat_vaqti'], alpha=0.3)
plt.xlabel('Hajm (litr)')
plt.ylabel('Xizmat vaqti (daqiqa)')
plt.show()

In [ ]:
filtered_df = df.copy()

In [ ]:
filtered_df['is_weekend'] = filtered_df['day_of_week'].isin(['Saturday', 'Sunday']).astype(int)

hourly_weekend = filtered_df.groupby(['is_weekend', 'hour'])['volume'].count().reset_index()


plt.figure(figsize=(12, 6))
sns.lineplot(data=hourly_weekend, x='hour', y='volume', hue='is_weekend', marker='o', linewidth=2)
plt.title('Ish kunlari (0) vs Dam olish kunlari (1) dagi mijozlar oqimi')
plt.xlabel('Kun soati')
plt.ylabel('Tranzaksiyalar soni')
plt.grid(True, alpha=0.3)
plt.legend(title='Dam olish kuni')
plt.show()

In [ ]:
filtered_df["vaqt_intervali"] = filtered_df["local_time"].dt.floor("30min").dt.time 

In [ ]:
filtered_df

In [ ]:
vaqt_intervali_stats = filtered_df.groupby(['branch_id', 'day_of_week', 'vaqt_intervali']).agg(
    faol_kolonkalar_shu_soatda=('dispenser_id', 'nunique'),  
    ortacha_hajm_shu_soatda=('volume', 'mean'), 
    jami_mashinalar_soni=('volume', 'count')
).reset_index()

vaqt_intervali_stats['ortacha_mashina_per_kolonka_shu_soatda'] = (
    vaqt_intervali_stats['jami_mashinalar_soni'] / vaqt_intervali_stats['faol_kolonkalar_shu_soatda']
)


vaqt_intervali_stats['is_weekend'] = vaqt_intervali_stats['day_of_week'].isin(['Saturday', 'Sunday']).astype(int)


In [ ]:
vaqt_intervali_stats

In [ ]:
vaqt_intervali_stats = vaqt_intervali_stats.sort_values(by=['branch_id', 'day_of_week', 'vaqt_intervali']).reset_index(drop=True)

vaqt_intervali_stats['mashina_soni_1_kun_oldin'] = (
    vaqt_intervali_stats.groupby(['branch_id', 'day_of_week'])['jami_mashinalar_soni'].shift(1)
)
vaqt_intervali_stats['mashina_soni_1_hafta_oldin'] = (
    vaqt_intervali_stats.groupby(['branch_id', 'day_of_week'])['jami_mashinalar_soni'].shift(7)
)


final_features = vaqt_intervali_stats[[
    'branch_id',
    'day_of_week',
    'vaqt_intervali', 
    'is_weekend',
    'faol_kolonkalar_shu_soatda', 
    'ortacha_mashina_per_kolonka_shu_soatda',
    'ortacha_hajm_shu_soatda',
]]

final_features.head(10)

In [ ]:
final_features.isnull().sum()

In [ ]:
final_features.count()

In [ ]:
final_features.shape

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df['volume'].value_counts()

In [ ]:
kam_hajmlar = df[df['volume'] < 5]['volume'].value_counts().sort_index()
print(kam_hajmlar)

In [ ]:
for i in [0.1, 0.5, 1, 2, 3, 5]:
    print(f"Datasetning eng pastki {i}% qismidagi chegara: {df['volume'].quantile(i/100):.3f} litr")

In [ ]:
for i in [0.0, 0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.4]:
    print(f"Datasetning eng chekka {i}% qismidagi qiymat: {df['volume'].quantile(i/100):.3f} litr")

In [ ]:
foizlar = np.linspace(0, 10, 50)  
qiymatlar = [df['volume'].quantile(f / 100) for f in foizlar]

plt.figure(figsize=(10, 5))
plt.plot(foizlar, qiymatlar, marker='.', color='darkviolet', linewidth=2)

plt.title("Hajm (Volume) ustunining pastki 10% lik Persentil Grafigi", fontsize=13, fontweight='bold')
plt.xlabel("Persentil foizi (%)", fontsize=11)
plt.ylabel("Yoqilg'i hajmi (Litr)", fontsize=11)
plt.grid(True, linestyle=':', alpha=0.6)


plt.show()

In [ ]:
micro_volumes = df[df['volume'] < 2.8]

total_rows = len(df)
micro_rows = len(micro_volumes)
percent = (micro_rows / total_rows) * 100

print(f"Umumiy qatorlar soni: {total_rows}")
print(f"3 litrdan kam tranzaksiyalar soni: {micro_rows} ta")
print(percent)


micro_volumes.head(10)

In [ ]:
df.drop(index=micro_volumes.index, inplace=True)

In [ ]:
max_100 = df[df['volume'] > 100]
max_150 = df[df['volume'] > 150]
max_200 = df[df['volume'] > 200]
max_300 = df[df['volume'] > 300]

total_rows = len(df)

print(f"Umumiy qatorlar soni: {total_rows}\n")
print(f"100 litrdan katta tranzaksiyalar: {len(max_100)} ta ")
print(f"150 litrdan katta tranzaksiyalar: {len(max_150)} ta ")
print(f"200 litrdan katta tranzaksiyalar: {len(max_200)} ta ")
print(f"300 litrdan katta tranzaksiyalar: {len(max_300)} ta ")

df.sort_values(by='volume', ascending=False).head(10)

In [ ]:
df.drop(index=max_200.index, inplace=True)

In [ ]:
df.shape

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.boxplot(x=df['volume'], ax=axes[0], color='skyblue')
axes[0].set_title("Volume ustuni - Boxplot (Tozalangan)")
axes[0].set_xlabel("Litr (Volume)")

sns.histplot(df['volume'], bins=50, kde=True, ax=axes[1], color='salmon')
axes[1].set_title("Volume ustuni - Tarqalish (Tozalangan)")
axes[1].set_xlabel("Litr (Volume)")

plt.tight_layout()
plt.show()

In [ ]:
day_map = {'Monday':0, 'Tuesday':1, 'Wednesday':2, 'Thursday':3, 'Friday':4, 'Saturday':5, 'Sunday':6}
df['day_of_week_encoded'] = df['day_of_week'].map(day_map)

In [ ]:
df['date_only'] = df['local_time'].dt.date

In [ ]:
hourly_data = df.groupby(['branch_id', 'date_only', 'month', 'day_of_week_encoded', 'hour']).agg(
    mashina_soni=('volume', 'count')
).reset_index()

In [ ]:
branch_daily_stats = hourly_data.groupby(['branch_id', 'date_only'])['mashina_soni'].agg(['max', 'min']).reset_index()
branch_daily_stats.columns = ['branch_id', 'date_only', 'daily_max_soni', 'daily_min_soni']

hourly_data = pd.merge(hourly_data, branch_daily_stats, on=['branch_id', 'date_only'], how='left')

hourly_data['target_usul_1'] = hourly_data['mashina_soni'] / hourly_data['daily_max_soni']

hourly_data['target_usul_2'] = (hourly_data['mashina_soni'] - hourly_data['daily_min_soni']) / (hourly_data['daily_max_soni'] - hourly_data['daily_min_soni'] + 1e-5)

In [ ]:
hourly_data['datetime_full'] = pd.to_datetime(hourly_data['date_only'].astype(str)) + pd.to_timedelta(hourly_data['hour'], unit='h')

malumot1 = hourly_data[['branch_id', 'datetime_full', 'target_usul_1']].copy()
malumot2 = hourly_data[['branch_id', 'datetime_full', 'target_usul_2']].copy()

#--- TARGET 1

# 1 kun oldin 
malumot1_1kun = malumot1.copy()
malumot1_1kun['datetime_full'] = malumot1_1kun['datetime_full'] + pd.Timedelta(days=1)
malumot1_1kun = malumot1_1kun.rename(columns={'target_usul_1': 'bir_kun_oldin_usul_1'})

# 2 kun oldin 
malumot1_2kun = malumot1.copy()
malumot1_2kun['datetime_full'] = malumot1_2kun['datetime_full'] + pd.Timedelta(days=2)
malumot1_2kun = malumot1_2kun.rename(columns={'target_usul_1': 'ikki_kun_oldin_usul_1'})

# 3 kun oldin 
malumot1_3kun = malumot1.copy()
malumot1_3kun['datetime_full'] = malumot1_3kun['datetime_full'] + pd.Timedelta(days=3)
malumot1_3kun = malumot1_3kun.rename(columns={'target_usul_1': 'uch_kun_oldin_usul_1'})

# 1 hafta oldin 
malumot1_1hafta = malumot1.copy()
malumot1_1hafta['datetime_full'] = malumot1_1hafta['datetime_full'] + pd.Timedelta(days=7)
malumot1_1hafta = malumot1_1hafta.rename(columns={'target_usul_1': 'bir_hafta_oldin_usul_1'})


# --- TARGET 2

# 1 kun oldin
malumot2_1kun = malumot2.copy()
malumot2_1kun['datetime_full'] = malumot2_1kun['datetime_full'] + pd.Timedelta(days=1)
malumot2_1kun = malumot2_1kun.rename(columns={'target_usul_2': 'bir_kun_oldin_usul_2'})

# 2 kun oldin
malumot2_2kun = malumot2.copy()
malumot2_2kun['datetime_full'] = malumot2_2kun['datetime_full'] + pd.Timedelta(days=2)
malumot2_2kun = malumot2_2kun.rename(columns={'target_usul_2': 'ikki_kun_oldin_usul_2'})

# 3 kun oldin
malumot2_3kun = malumot2.copy()
malumot2_3kun['datetime_full'] = malumot2_3kun['datetime_full'] + pd.Timedelta(days=3)
malumot2_3kun = mal_col = malumot2.copy()
malumot2_3kun['datetime_full'] = malumot2_3kun['datetime_full'] + pd.Timedelta(days=3)
malumot2_3kun = malumot2_3kun.rename(columns={'target_usul_2': 'uch_kun_oldin_usul_2'})

# 1 hafta oldin
malumot2_1hafta = malumot2.copy()
malumot2_1hafta['datetime_full'] = malumot2_1hafta['datetime_full'] + pd.Timedelta(days=7)
malumot2_1hafta = malumot2_1hafta.rename(columns={'target_usul_2': 'bir_hafta_oldin_usul_2'})


# Merging

# Target 1 
hourly_data = hourly_data.merge(malumot1_1kun, on=['branch_id', 'datetime_full'], how='left')
hourly_data = hourly_data.merge(malumot1_2kun, on=['branch_id', 'datetime_full'], how='left')
hourly_data = hourly_data.merge(malumot1_3kun, on=['branch_id', 'datetime_full'], how='left')
hourly_data = hourly_data.merge(malumot1_1hafta, on=['branch_id', 'datetime_full'], how='left')

# Target 2
hourly_data = hourly_data.merge(malumot2_1kun, on=['branch_id', 'datetime_full'], how='left')
hourly_data = hourly_data.merge(malumot2_2kun, on=['branch_id', 'datetime_full'], how='left')
hourly_data = hourly_data.merge(malumot2_3kun, on=['branch_id', 'datetime_full'], how='left')
hourly_data = hourly_data.merge(malumot2_1hafta, on=['branch_id', 'datetime_full'], how='left')

# NaN qiymatlarni to'dirish

lag_columns = [
    'bir_kun_oldin_usul_1', 'ikki_kun_oldin_usul_1', 'uch_kun_oldin_usul_1', 'bir_hafta_oldin_usul_1',
    'bir_kun_oldin_usul_2', 'ikki_kun_oldin_usul_2', 'uch_kun_oldin_usul_2', 'bir_hafta_oldin_usul_2'
]

hourly_data = hourly_data.sort_values(by=['branch_id', 'datetime_full']).reset_index(drop=True)

for col in lag_columns:
    hourly_data[col] = hourly_data.groupby('branch_id')[col].bfill().ffill()

In [ ]:
hourly_data = hourly_data.sort_values(['branch_id', 'hour', 'datetime_full']).reset_index(drop=True)

# TARGET 1 
hourly_data['oxirgi_7kun_ortacha_usul_1'] = (
    hourly_data.groupby(['branch_id', 'hour'])['target_usul_1']
    .transform(lambda x: x.shift(1).rolling(window=7, min_periods=1).mean())
)
hourly_data['oxirgi_7kun_ortacha_usul_1'] = hourly_data['oxirgi_7kun_ortacha_usul_1'].ffill().bfill()


# TARGET 2 
hourly_data['oxirgi_7kun_ortacha_usul_2'] = (
    hourly_data.groupby(['branch_id', 'hour'])['target_usul_2']
    .transform(lambda x: x.shift(1).rolling(window=7, min_periods=1).mean())
)
hourly_data['oxirgi_7kun_ortacha_usul_2'] = hourly_data['oxirgi_7kun_ortacha_usul_2'].ffill().bfill()


hourly_data = hourly_data.sort_values(['branch_id', 'datetime_full']).reset_index(drop=True)

In [ ]:
dataset_solishtirish = hourly_data[[
    'branch_id', 'month', 'day_of_week_encoded', 'hour', 'mashina_soni', 
    'target_usul_1', 
    'target_usul_2',  
    # Target 1 uchun 
    'bir_kun_oldin_usul_1', 
    'ikki_kun_oldin_usul_1',
    'uch_kun_oldin_usul_1',
    'bir_hafta_oldin_usul_1',
    'oxirgi_7kun_ortacha_usul_1',
    # Target 2 uchun 
    'bir_kun_oldin_usul_2', 
    'ikki_kun_oldin_usul_2',
    'uch_kun_oldin_usul_2',
    'bir_hafta_oldin_usul_2',
    'oxirgi_7kun_ortacha_usul_2'
]]

In [ ]:
dataset_solishtirish

In [ ]:
dataset = dataset_solishtirish.copy()

In [ ]:
month_map = {
    'January': 1, 'February': 2, 'March': 3, 'April': 4, 'May': 5, 'June': 6,
    'July': 7, 'August': 8, 'September': 9, 'October': 10, 'November': 11, 'December': 12
}
dataset['month_encoded'] = dataset['month'].map(month_map)

In [ ]:
dataset

In [ ]:
aniq_sana = pd.to_datetime('2025-04-24').date()


namuna_kun = dataset_solishtirish[hourly_data['date_only'] == aniq_sana].sort_values(by='hour').reset_index(drop=True)

to_liq_soatlar = pd.DataFrame({'hour': range(0, 24)})

namuna_kun = pd.merge(to_liq_soatlar, namuna_kun, on='hour', how='left')

namuna_kun['target_usul_1'] = namuna_kun['target_usul_1'].fillna(0.0)
namuna_kun['target_usul_2'] = namuna_kun['target_usul_2'].fillna(0.0)


plt.figure(figsize=(12, 6))

plt.plot(namuna_kun['hour'], namuna_kun['target_usul_1'], 
         marker='o', linestyle='-', linewidth=2.5, color='royalblue', 
         label="1-Usul: Max-Scaling")

plt.plot(namuna_kun['hour'], namuna_kun['target_usul_2'], 
         marker='s', linestyle='--', linewidth=2, color='darkorange', 
         label="2-Usul: Min-Max Scaling")

plt.title(f"Zapravkaning To'liq 24 Soatlik ({aniq_sana}) Bandlik Grafigi", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Sutka soatlari (X o'qi — 24 Hour)", fontsize=12)
plt.ylabel("Bandlik Koeffitsiyenti (Y o'qi — Target)", fontsize=12)

plt.xticks(range(0, 24))
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(fontsize=11, loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
X1_cols = [
    'month_encoded', 'day_of_week_encoded', 'hour', 
    'bir_kun_oldin_usul_1', 'ikki_kun_oldin_usul_1', 
    'uch_kun_oldin_usul_1', 'bir_hafta_oldin_usul_1'
]

X1 = dataset[X1_cols]
y1 = dataset['target_usul_1']


X2_cols = [
    'month_encoded', 'day_of_week_encoded', 'hour', 
    'bir_kun_oldin_usul_2', 'ikki_kun_oldin_usul_2', 
    'uch_kun_oldin_usul_2', 'bir_hafta_oldin_usul_2', 
]

X2 = dataset[X2_cols]
y2 = dataset['target_usul_2']

In [ ]:
barcha_sanalar = sorted(hourly_data['date_only'].unique())
kesim_indeks = int(len(barcha_sanalar) * 0.8) 
kesim_sana = barcha_sanalar[kesim_indeks]
print("Kesim sanasi:", kesim_sana)

In [ ]:
kesim_sana = pd.to_datetime('2025-08-11').date()
train_mask = hourly_data['date_only'] < kesim_sana
test_mask = hourly_data['date_only'] >= kesim_sana

X1_train, X1_test = X1[train_mask], X1[test_mask]
y1_train, y1_test = y_1[train_mask], y_1[test_mask]

X2_train, X2_test = X2[train_mask], X2[test_mask]
y2_train, y2_test = y_2[train_mask], y_2[test_mask]

In [ ]:
print("X1_train:", X1_train.shape)
print("X1_test:", X1_test.shape)
print("X2_train:", X2_train.shape)
print("X2_test:", X2_test.shape)

In [ ]:
cat_features = ['day_of_week_encoded']

# Target 1
cat_model_1 = CatBoostRegressor(iterations=78, depth=6, learning_rate=0.1, random_seed=42, verbose=False)
cat_model_1.fit(X1_train, y1_train, cat_features=cat_features)

rf_model_1 = RandomForestRegressor(n_estimators=400, max_depth=6, random_state=42)
rf_model_1.fit(X1_train, y1_train)

xgb_model_1 = XGBRegressor(n_estimators=80, max_depth=3, learning_rate=0.1, random_state=42)
xgb_model_1.fit(X1_train, y1_train)

In [ ]:
# Target 2
cat_model_2 = CatBoostRegressor(iterations=91, depth=6, learning_rate=0.1, random_seed=42, verbose=False)
cat_model_2.fit(X2_train, y2_train, cat_features=cat_features)

rf_model_2 = RandomForestRegressor(n_estimators=400, max_depth=6, random_state=42)
rf_model_2.fit(X2_train, y2_train)

xgb_model_2 = XGBRegressor(n_estimators=98, max_depth=3, learning_rate=0.1, random_state=42)
xgb_model_2.fit(X2_train, y2_train)

In [ ]:
cat_preds_1 = cat_model_1.predict(X1_test)
rf_preds_1 = rf_model_1.predict(X1_test)
xgb_preds_1 = xgb_model_1.predict(X1_test)

natijalar_1 = {}
for name, preds in [('CatBoost', cat_preds_1), ('RandomForest', rf_preds_1), ('XGBoost', xgb_preds_1)]:
    mae = mean_absolute_error(y1_test, preds)
    rmse = np.sqrt(mean_squared_error(y1_test, preds))
    r2 = r2_score(y1_test, preds)
    natijalar_1[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2}

natijalar_1_df = pd.DataFrame(natijalar_1).T
natijalar_1_df

In [ ]:
cat_preds_2 = cat_model_2.predict(X2_test)
rf_preds_2 = rf_model_2.predict(X2_test)
xgb_preds_2 = xgb_model_2.predict(X2_test)

natijalar_2 = {}
for name, preds in [('CatBoost', cat_preds_2), ('RandomForest', rf_preds_2), ('XGBoost', xgb_preds_2)]:
    mae = mean_absolute_error(y2_test, preds)
    rmse = np.sqrt(mean_squared_error(y2_test, preds))
    r2 = r2_score(y2_test, preds)
    natijalar_2[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2}

natijalar_2_df = pd.DataFrame(natijalar_2).T
natijalar_2_df

In [ ]:
# Target 1
for name, model in [('CatBoost', cat_model_1), ('RandomForest', rf_model_1), ('XGBoost', xgb_model_1)]:
    train_preds = model.predict(X1_train)
    test_preds = model.predict(X1_test)
    train_r1 = r2_score(y1_train, train_preds)
    test_r1 = r2_score(y1_test, test_preds)
    print(f"{name}: Train R²={train_r1:.4f}, Test R²={test_r1:.4f}, Farq={train_r1-test_r1:.4f}")

In [ ]:
# Target 2
for name, model in [('CatBoost', cat_model_2), ('RandomForest', rf_model_2), ('XGBoost', xgb_model_2)]:
    train_preds = model.predict(X2_train)
    test_preds = model.predict(X2_test)
    train_r2 = r2_score(y2_train, train_preds)
    test_r2 = r2_score(y2_test, test_preds)
    print(f"{name}: Train R²={train_r2:.4f}, Test R²={test_r2:.4f}, Farq={train_r2-test_r2:.4f}")

In [ ]:
# Oxirgi 7 kunlik o'rtacha

baseline_preds_1 = dataset.loc[X_test.index, 'oxirgi_7kun_ortacha_usul_1']

b1_mae = mean_absolute_error(y1_test, baseline_preds_1)
b1_rmse = np.sqrt(mean_squared_error(y1_test, baseline_preds_1))
b1_r2 = r2_score(y1_test, baseline_preds_1)

baseline_preds_2 = dataset.loc[X_test.index, 'oxirgi_7kun_ortacha_usul_2']

b2_mae = mean_absolute_error(y2_test, baseline_preds_2)
b2_rmse = np.sqrt(mean_squared_error(y2_test, baseline_preds_2))
b2_r2 = r2_score(y2_test, baseline_preds_2)

print("-*-*- TARGET 1 BASELINE (7 kunlik o'rtacha) -*-*-")
print("MAE  :", round(b1_mae, 4))
print("RMSE :", round(b1_rmse, 4))
print("R2   :", round(b1_r2, 4))

print("\n-*-*- TARGET 2 BASELINE (7 kunlik o'rtacha) -*-*-")
print("MAE  :", round(b2_mae, 4))
print("RMSE :", round(b2_rmse, 4))
print("R2   :", round(b2_r2, 4))

In [ ]:
importances = rf_model_1.feature_importances_

feature_names = X1_train.columns
feature_imp_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=True) 

plt.figure(figsize=(10, 6))
colors = plt.cm.viridis(np.linspace(0.3, 0.8, len(feature_imp_df)))

plt.barh(feature_imp_df['Feature'], feature_imp_df['Importance'], color=colors, edgecolor='black')
plt.xlabel('Muhimlik darajasi (Importance Score)', fontsize=12)
plt.ylabel('Featurelar (Ustunlar)', fontsize=12)
plt.title('Random Forest - Feature Importance', fontsize=14, fontweight='bold')
plt.grid(axis='x', linestyle='--', alpha=0.7)

for index, value in enumerate(feature_imp_df['Importance']):
    plt.text(value + 0.005, index, f'{value:.4f}', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
importances = rf_model_2.feature_importances_

feature_names = X2_train.columns
feature_imp_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=True) 

plt.figure(figsize=(10, 6))
colors = plt.cm.viridis(np.linspace(0.3, 0.8, len(feature_imp_df)))

plt.barh(feature_imp_df['Feature'], feature_imp_df['Importance'], color=colors, edgecolor='black')
plt.xlabel('Muhimlik darajasi (Importance Score)', fontsize=12)
plt.ylabel('Featurelar (Ustunlar)', fontsize=12)
plt.title('Random Forest - Feature Importance', fontsize=14, fontweight='bold')
plt.grid(axis='x', linestyle='--', alpha=0.7)

for index, value in enumerate(feature_imp_df['Importance']):
    plt.text(value + 0.005, index, f'{value:.4f}', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 12))

modellar = [('CatBoost', cat_preds_2), ('RandomForest', rf_preds_2), ('XGBoost', xgb_preds_2)]

for i, (name, preds) in enumerate(modellar):
    axes[i].plot(y2_test.values[:100], label='Haqiqiy', marker='o')
    axes[i].plot(preds[:100], label=f'Bashorat ({name})', marker='x')
    axes[i].legend()
    axes[i].set_title(f'Haqiqiy vs Bashorat - {name}')
    axes[i].set_ylabel('Bandlik darajasi')

axes[2].set_xlabel('Test namunasi')
plt.tight_layout()
plt.show()